In [1]:
import pandas as pd
import numpy as np

# LIBRERIAS DE VISUALIZACION QUE GENERAN GRAFICOS INTERACTIVOS
import plotly.graph_objects as go

# LIBRERIAS DE SKLEARN
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

In [2]:
df_salud = pd.read_csv("../DATASETS/df_enfermedades_cardiacas.csv")
df_salud

,Tipo_Dolor_Pecho,Frecuencia_Cardiaca_Maxima,Angina_Ejercicio,Depresion_ST_Ejercicio,Pendiente_Segmento_ST,Numero_Vasos_Filtrados,Enfermedad_Cardiaca
0,0,168,0,1.0,2,2,0
1,0,155,1,3.1,0,0,0
2,0,125,1,2.6,0,0,0
3,0,161,0,0.0,2,1,0
4,0,106,0,1.9,1,3,0
...,...,...,...,...,...,...,...
1020,1,164,1,0.0,2,0,1
1021,0,141,1,2.8,1,1,0
1022,0,118,1,1.0,1,1,0
1023,0,159,0,0.0,2,0,1


In [3]:
# SE INICIA DIVIDIENDO EL DATASET EN CONJUNTOS DE ENTRENAMIENTO Y PRUEBA
X = df_salud.drop("Enfermedad_Cardiaca", axis=1)
y = df_salud["Enfermedad_Cardiaca"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [4]:
# CALCULAR LOS MEJORES K VECINOS PARA KMEANS
k_values = range(1, 11)
K_scores = []

for i in k_values:

    # CALCULAMOS EL ACURACY SCORE PARA CADA K
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)

    accuracy = accuracy_score(y_train, knn.predict(X_train))
    K_scores.append(accuracy)

k_df = pd.DataFrame({"K_Vecino": k_values, "Accuracy Score": K_scores}).reset_index(drop=True)
k_df

,K_Vecino,Accuracy Score
0,1,1.000000
1,2,0.994421
2,3,0.981869
3,4,0.938633
4,5,0.909344
5,6,0.892608
6,7,0.860530
7,8,0.860530
8,9,0.860530
9,10,0.852162


In [5]:
# TRAEMOS EN UNA VARIABLE EL K_VECUNO CON EL MEJOR ACURACY SCORE, EXCLUYENDO EL K=1
best_k = k_df[k_df["K_Vecino"] != 1].sort_values(by="Accuracy Score", ascending=False).iloc[0]["K_Vecino"]

# IMPRIMIR EL MEJOR K VECINO EN FORMATO ENTERO, SIN DECIMALES
print(f"El mejor K Vecino para el modelo KNN es: {int(best_k)}")

El mejor K Vecino para el modelo KNN es: 2


**SE OBSERVAN QUE LOS MEJORES K VECINOS PARA EL MODELO SE ENCUENTRA ENTRE 2 y 3 K VECINOS**

In [6]:
# SE GRAFICA LOS MEJORES K VECINOS CON PLOTLY
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=list(k_values), y=K_scores, mode="lines+markers", name="Accuracy Score")
)
fig.update_layout(
    title="Accuracy Score para diferentes valores de K",
    xaxis_title="Número de K Vecinos",
    yaxis_title="Accuracy Score",
    xaxis=dict(tickmode="linear"),
    template="ggplot2"
)
fig.show()

In [7]:
knn_final = KNeighborsClassifier(n_neighbors=2)
knn_final.fit(X_train, y_train)
y_pred = knn_final.predict(X_train)
accuracy_final = accuracy_score(y_train, y_pred)
print("Accuracy Score del modelo KNN con K=2: {:.4f}".format(accuracy_final))

Accuracy Score del modelo KNN con K=2: 0.9944


**CONCLUSIÓN:** SE OBSERVA QUE EL MODELO TIENE UN ACCURACY MUY ALTO, LO QUE INDICA QUE EL 99% DE LAS OCASIONES PREDICE MUY BIEN. 

In [8]:
# SE PROCEDE A GRAFICAR LA MATRIZ DE CONFUSION PARA EL MODELO KNN CON K=2 EN PLOTLY
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_train, y_pred)
cm_df = pd.DataFrame(cm, index=["No Enfermedad", "Enfermedad"], columns=["Predicho No Enfermedad", "Predicho Enfermedad"])
fig_cm = go.Figure(
    data=go.Heatmap(
        z=cm_df.values,
        x=cm_df.columns,
        y=cm_df.index,
        colorscale="Blues",
        showscale=True,
    )
)
fig_cm.show()

**La matriz de confusión muestra un excelente desempeño del modelo. Se identificaron 373 casos de pacientes con enfermedad cardíaca que fueron clasificados correctamente (verdaderos positivos). Sin embargo, se presentaron 4 casos en los que el modelo predijo que no existía enfermedad cuando en realidad sí estaba presente (falsos negativos).**

**Por otro lado, el modelo clasificó correctamente 340 casos de pacientes sin enfermedad cardíaca (verdaderos negativos) y no registró falsos positivos, es decir, no hubo casos en los que se predijera enfermedad en pacientes que realmente estaban sanos.**

In [9]:
# COMPARAR Y_REAL VS Y_PREDICT
comparasion_df = pd.DataFrame({"Real": y_train, "Predicho": y_pred})

# SE CREA UNA NUEVA COLUMNA DONDE COMPARA LOS VALORES REALES CON LOS PREDICHOS Y ASIGNA CORRECTO EN CASO DE QUE LA PREDICCION SEA CORRECTA O INCORRECTO EN CASO CONTRARIO
comparasion_df["Resultado"] = np.where(comparasion_df["Real"] == comparasion_df["Predicho"], "Correcto", "Incorrecto")
comparasion_df

,Real,Predicho,Resultado
1020,1,1,Correcto
479,0,0,Correcto
227,1,1,Correcto
910,0,0,Correcto
362,1,1,Correcto
...,...,...,...
700,1,1,Correcto
71,0,0,Correcto
106,0,0,Correcto
270,1,1,Correcto


In [10]:
# SE CUENTA LA CANTIDAD DE PREDICCIONES CORRECTAS E INCORRECTAS
comparasion_df["Resultado"].value_counts()

Resultado
Correcto      713
Incorrecto      4
Name: count, dtype: int64

In [11]:
# SE PROCEDE A EXPORTAR EL DATAFRAME DE COMPARACION A UN ARCHIVO CSV
ruta_exportacion = "../PREDICCIONES/COMPARACION_KNN2.xlsx"

comparasion_df.to_excel(ruta_exportacion, index=False)
print("EL DATAFRAME DE COMPARACION SE HA EXPORTADO CORRECTAMENTE A LA RUTA: {}".format(ruta_exportacion))

EL DATAFRAME DE COMPARACION SE HA EXPORTADO CORRECTAMENTE A LA RUTA: ../PREDICCIONES/COMPARACION_KNN2.xlsx


**PRUEBAS CON K VECINOS = 3**

In [12]:
knn_final_prueba3 = KNeighborsClassifier(n_neighbors=3)
knn_final_prueba3.fit(X_train, y_train)
y_pred = knn_final_prueba3.predict(X_train)
accuracy_final = accuracy_score(y_train, y_pred)
print("Accuracy Score del modelo KNN con K=3: {:.4f}".format(accuracy_final))

Accuracy Score del modelo KNN con K=3: 0.9819


In [13]:
# SE CREA LA MATRIZ DE CONFUSION PARA EL MODELO KNN CON K=3 EN PLOTLY
cm = confusion_matrix(y_train, y_pred)
cm_df = pd.DataFrame(cm, index=["No Enfermedad", "Enfermedad"], columns=["Predicho No Enfermedad", "Predicho Enfermedad"])
fig_cm_kn3 = go.Figure(
    data=go.Heatmap(
        z=cm_df.values,
        x=cm_df.columns,
        y=cm_df.index,
        colorscale="Blues",
        showscale=True,
    )
)
fig_cm_kn3.show()

**La matriz de confusión muestra un buen desempeño del modelo, no tan destacado como implementar 2 k vecinos. Se identificaron 373 casos de pacientes con enfermedad cardíaca que fueron clasificados correctamente (verdaderos positivos). Sin embargo, se presentaron 4 casos en los que el modelo predijo que no existía enfermedad cuando en realidad sí estaba presente (falsos negativos).**

**Por otro lado, el modelo clasificó correctamente 331 casos de pacientes sin enfermedad cardíaca (verdaderos negativos) y registró 9 falsos positivos, es decir, 9 casos en los que se predijera enfermedad en pacientes que realmente estaban sanos.**